In [2]:
import os
import httpx
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma   
load_dotenv()

True

In [3]:
def get_transcript(video_id: str) -> dict: #key is content  and content contains a list of dictionaries with keys as lang text offset duration
    SUPADATA_API_KEY = os.environ.get("SUPADATA_API_KEY")
    
    response = httpx.get(
        "https://api.supadata.ai/v1/youtube/transcript",
        params={"videoId": video_id},
        headers={"x-api-key": SUPADATA_API_KEY},
        timeout=60.0,
    )
    
    if response.status_code != 200:
        raise Exception(f"Supadata error: {response.status_code} - {response.text}")
    
    data = response.json()
    
    return data

data = get_transcript("WDSRXu4cJbM")

In [4]:
docs = []
current_text = ""
start_offset = None

for segment in data["content"]:
    if start_offset is None:
        start_offset = segment["offset"]

    current_text += " " + segment["text"]

    # reate a chunk every 1000 characters
    if len(current_text) >= 1000:
        docs.append(
            Document(
                page_content=current_text.strip(),
                metadata={"offset": start_offset}
            )
        )
        current_text = ""
        start_offset = None

# remaining text
if current_text:
    docs.append(
        Document(
            page_content=current_text.strip(),
            metadata={"offset": start_offset}
        )
    )

docs

[Document(metadata={'offset': 0}, page_content='आप ये चीज नोटिस करोगे आपके सारे जितने भी फेमस आप मुंह तोड़ बुलतोड़ जो यूट्यूबर हैं जो वैसे तो ताबड़तोड़ गालियां बकते हैं और स्ट्रांग ओपिनियंस रखते हैं और तुम्हारा पूरा उन्होंने आठ साल का करियर है किसी का छ साल का करियर है भाई बैठे हैं और तुम्हें पता है सब करोड़पति हैं अपने घर दिखाते हैं भाई हम कितने करोड़पति हैं और आपको मरे रहते हैं कि आप हमसे सबसे आप हमारे परिवार हैं आप हमारे सबसे छोटे भाई हैं छोटी बहनें हैं मैं आपसे इतना प्यार करता हूं आपसे इतना प्यार करता हूं मैं आपसे इतना प्यार करता हूं करोड़ों ठूसे हुए हैं इन लोगों ने दो कौड़ी की चीजें बेचने में स्पॉनसरशिप के नाम पे ये हटते नहीं है पीछे आपको दुनिया में कोई भी 2 करोड़ की चीज के पीछे इनको जिनको 20 से 1 करोड़ लाख से 1 करोड़ के बीच में कमाते हैं पर स्पोंसरशिप में आपको बता देता हूं असली रेट क्या है उनको वो बेचने में कोई फर्क नहीं पड़ता उनको पता है उनका पेट पेट भरना है उनका पैसा मोटा हो रहा है लेकिन जब बाकी आपकी जिंदगी की बात आती है आपकी राइट्स की बात आती है तो ये चुप्पार लेते हैं निकल लेत

In [13]:
vector_store = Chroma(
    embedding_function=OpenAIEmbeddings(
        model="openai/text-embedding-3-large",
        dimensions=1536,
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url="https://openrouter.ai/api/v1"
    ),
    persist_directory='my_chroma_db',
    collection_name='videos_transcript_chunks_1536'
)


vector_store.add_documents(docs)

['1cc715bc-9a67-4aed-8894-eccafc94fc99',
 'd3568b1c-014a-4540-8c05-b0647ad4d16b',
 'c2b99b2a-e304-4b4e-b04c-1930d4ec8c66',
 '7ebc8655-974e-4098-921c-8f5144766c72',
 'ca72bbec-63e5-4d58-b647-3ff6822e7bd7',
 '787c6e3c-9f63-4661-a7b8-0014b8f57d89',
 '7a1dbebe-9aaa-4cfb-bc4f-a950697dad52',
 '392edb2f-3e82-4182-a73c-0785026a7f69',
 '278c313c-8220-4249-bc07-52c8afb14474']

In [14]:
vector_store.similarity_search(
    query="where did he talks about transition period between 25 to 28 where you will stop watching these youtube creators",
    k=3
)

[Document(id='7ebc8655-974e-4098-921c-8f5144766c72', metadata={'offset': 167760}, page_content='जिनको थ्रू शियर लक क्योंकि इन्होंने बहुत पहले YouTube चैनल स्टार्ट कर दिया। उस टाइम पर इतने ज्यादा यूटबर या उतने ज्यादा क्रिएटर्स नहीं थे। यही सब लोग अगर इस वक्त 2026 में शुरू हुए थे जीरो के साथ तो मे बी इनमें से 95% कभी कोई फेमस भी ना हो पाते। लेट्स बी वेरी रियल। इनके में कोई ऐसा खतरनाक स्पेशली यूनिक खतरा कोई टैलेंट नहीं है। कोई इनकी स्क्रिप्ट में ऐसी खतरनाक चीज नहीं है। सिवाय 100 में से पांच या तीन को छोड़ के जो आज भी अगर 2026 में शुरू होते एग्जैक्टली उसी उम्र में 15, 16, 17, 18 जब भी शुरू हुए तो आज भी अपना करियर बना पाते। मे बी 95% अगर मुझे प्रेडिक्शन रखना पड़े या बेट रखना पड़े मैं रखूंगा 95% फेलियर रेट है कि इनमें से ये लोग कभी रेप्लिकेट नहीं कर पाएंगे अपना फेम। वो टाइम अलग था। ये लोग अर्ली स्टार्ट हो गए। इंटरनेट अलग था, कंसोलिडेटेड था। जो भी पावर थी व्यूज की YouTube पे तो व्यू YouTube पे जो भी कूद पाया पहले बहुत हद तक बहुत सारे लोग अपने को उठा पाए और आज भी उन्होंने मोनोपोलाइज कर रखा है 